# ⚙️ Конвейер за обработка на данни (Data Processing Pipeline)

Този ноутбук демонстрира автоматизирания процес по подготовка на аудио данните от набора **ESC-50**. За да се спазят добрите инженерни практики, същинската логика по обработката е изнесена в модулни Python скриптове в папката `src/data_processing/`.

Тук ще заредим тези модули, ще разделим данните безопасно (без изтичане на информация) и ще тестваме пълния цикъл на трансформация от суров `.wav` файл до нормализиран 3D тензор, готов за нашата Конволюционна невронна мрежа (CNN).

## 1. Инициализация и импортиране на модулите
Дефинираме основните библиотеки и пътищата до суровите файлове.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
# Импортиране на библиотеките и твоите модули
import numpy as np
import matplotlib.pyplot as plt
import librosa.display
from tqdm import tqdm 

# Импортираш твоите функции (увери се, че пътищата отговарят на структурата на папките ти)
from src.data_processing.splitting import split_data
from src.data_processing.clean_audio import load_and_clean_audio
from src.data_processing.standardize_lenght import standardize_lenght
from src.data_processing.extract_mel import extract_mel_spectrogram
from src.data_processing.format_for_mel import format_for_mel

# Пътища до файловете
CSV_PATH = "../data/raw/ESC-50-master/meta/esc50.csv"
AUDIO_DIR = "../data/raw/ESC-50-master/audio/"

## 2. Разделяне на данните (Data Splitting)

За да предотвратим изтичане на данни (**Data Leakage**) и да гарантираме обективна оценка на модела, не смесваме всички файлове на случаен принцип. Използваме предварително дефинираните 5 фолда (folds) на ESC-50:
* **Train Set (Обучение):** Фолдове 1, 2 и 3 (60% от данните).
* **Validation Set (Валидация):** Фолд 4 (20%) – за настройка на хиперпараметрите по време на обучение.
* **Test Set (Тестване):** Фолд 5 (20%) – напълно непознати данни за финална оценка.

In [ ]:
# Разделяне на данните (Data Splitting)
train_df, val_df, test_df = split_data(CSV_PATH)

print(f"Тренировъчни файлове: {len(train_df)}")
print(f"Валидационни файлове: {len(val_df)}")
print(f"Тестови файлове: {len(test_df)}")

Тренировъчни файлове: 1200
Валидационни файлове: 400
Тестови файлове: 400


## 3. Дефиниране на конвейера (The Processing Pipeline)

Обединяваме нашите модулни функции в единен процес (`process_single_file`), който изпълнява следните 4 стъпки:
1. **Екстракция и изчистване (`load_and_clean_audio`):** Зарежда файла и премахва тишината в двата края (праг 20dB).
2. **Времева стандартизация (`standardize_lenght`):** Фиксира дължината до точно 2.0 секунди (чрез Zero-Padding или Center Cropping).
3. **Спектрална трансформация (`extract_mel_spectrogram`):** Генерира Мел-спектрограма (128 филтъра, до 8000 Hz) в логаритмична скала (децибели).
4. **ML Форматиране (`format_for_mel`):** Прилага *per-sample* нормализация (Min-Max в диапазона [0, 1]) и добавя цветови канал, за да се получи тензор, съвместим със CNN.

In [ ]:
# Дефиниране на обединяващата функция (The Pipeline)
def process_single_file(filename, audio_dir):
    """
    Прекарва един аудио файл през целия конвейер.
    """
    file_path = os.path.join(audio_dir, filename)
    
    # 1. Зареждане и изчистване на тишината
    y, sr = load_and_clean_audio(file_path, sr=22050, top_db=20)
    
    # 2. Стандартизиране до 2.0 секунди
    y_std = standardize_lenght(y, sr, targer_duration=2.0)
    
    # 3. Екстракция на Мел-спектрограма
    mel_db = extract_mel_spectrogram(y_std, sr, n_mels=128, fmax=8000)
    
    # 4. Форматиране и нормализация
    mel_tensor = format_for_mel(mel_db)
    
    return mel_tensor

## 4. Тестване на конвейера (Sanity Check)

За да се уверим, че всички модули работят в синхрон, ще извлечем първия файл от тренировъчния набор и ще го прекараме през конвейера. Ще проверим дали финалната форма на матрицата е правилна и дали стойностите са успешно нормализирани между 0 и 1.

In [ ]:
# Тестване на конвейера върху един файл (Sanity Check)
test_filename = train_df.iloc[0]['filename'] # Взимаме първия файл от тренировъчния сет
tensor = process_single_file(test_filename, AUDIO_DIR)

print(f"Оригинален файл: {test_filename}")
print(f"Форма на финалния тензор: {tensor.shape}") 
print(f"Минимална стойност: {tensor.min()} (Трябва да е 0.0)")
print(f"Максимална стойност: {tensor.max()} (Трябва да е 1.0)")

Оригинален файл: 1-100032-A-0.wav
Форма на финалния тензор: (1, 128, 87)
Минимална стойност: 0.0 (Трябва да е 0.0)
Максимална стойност: 1.0 (Трябва да е 1.0)
